In [0]:
%run ../lib/ingestion_functions

In [0]:
container_source = "novadrive-project"
directory_source = "raw_ans"
subdirectory_source = "sip"
container_target = "bronze"
directory_target = "sip"

file_source = "sip"
source_table = f"bronze_{file_source}"
source_schema = "bronze"
target_schema = "silver"
delta_table_name = f"silver_{file_source}"
id_field = "PORTE_OPERADORA || GR_MODALIDADE || COBERTURA || ID_TRIMESTRE || CONTRATACAO || ID_ITEM_ASST"
timestamp_field = "ID_TRIMESTRE"
flag_cdf = "1"

In [0]:
target_path = f"s3://novadrive-project/silver/{delta_table_name}"
schema_location = f"s3://novadrive-project/metastore/silver/{delta_table_name}_schema"
checkpoint_location = f"s3://novadrive-project/metastore/silver/{delta_table_name}_chk"

In [0]:
table_exists = spark.catalog.tableExists(f"dev.silver.{delta_table_name}")
if not table_exists:

    dbutils.fs.rm(checkpoint_location, True)
    dbutils.fs.rm(schema_location, True)

    print(f"Tabela {delta_table_name} não existe ou a carga é Full Load, criando...")

    bronzeSelect = f"""
            select * from dev.bronze.{source_table}
        """
    bronzeInfo = spark.sql(bronzeSelect)

    creator = DeltaTableCreator(spark)

    creator.create_table_with_cdf(
        df=bronzeInfo,
        catalog="dev",
        schema="silver",
        table=delta_table_name,
        path=target_path,
        mode="overwrite"  # sobrescreve dados no path se já existirem
    )

else:

    ingestor = IngestionCDF(
    spark=spark,
    data_format="delta",
    target_path=target_path,
    checkpoint_location=checkpoint_location,
    source_table = source_table,
    source_schema = source_schema,
    target_schema = target_schema,
    catalog="dev",
    schemaname="silver",
    id_field=id_field,
    timestamp_field=timestamp_field,
    tablename=delta_table_name
)

    
    print(f"Tabela {delta_table_name} já existe, executando CDF...")

    ingestor.executeLoadAndSaveCDF(delta_table_name)

In [0]:
# df = spark.read.table("dev.silver.silver_sip")  # ou apenas "schema.tabela" se for Hive Metastore simples

# df.coalesce(1).write.mode("overwrite").parquet(f"s3://novadrive-project/export/{delta_table_name}")